In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
%cd "/content/drive/MyDrive/Deep Learning/protein_dynamics_project"

/content/drive/MyDrive/Deep Learning/protein_dynamics_project


In [ ]:
# Runs all preprocessing, alignment, and saves aligned arrays
%run src/01_preprocessing.py

PROJECT_ROOT = /content/drive/MyDrive/Deep Learning/protein_dynamics_project
DATA_RAW  = /content/drive/MyDrive/Deep Learning/protein_dynamics_project/data/raw
PROCESSED = /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed

Files in data/raw:
 - coordinates.csv
 - forces.csv
 - velocity.csv
Using:
 - coordinates.csv
 - forces.csv
 - velocity.csv
Reading raw files...
Raw shapes (rows, cols_without_time):
coords : (50001, 414)
vels   : (50001, 414)
forces : (50001, 414)

Aligned length: 50001
n_atoms inferred: 138
Final shapes:
coords : (50001, 138, 3)
vels   : (50001, 138, 3)
forces : (50001, 138, 3)

Saved:
 - /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/coords_aligned.npy
 - /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/vels_aligned.npy
 - /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/forces_aligned.npy


In [ ]:
!python src/02_make_windows.py --window 20

Loading aligned arrays...
Shapes:
coords : (50001, 138, 3)
vels   : (50001, 138, 3)
forces : (50001, 138, 3)
Per-timestep feature dims:
X_source: (50001, 828) (per step: 828 )
y_source: (50001, 1242) (per step: 1242 )
Building sliding windows...
DONE.
X shape: (49981, 20, 828)
y shape: (49981, 1242)
Saved to: /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/windows_w20


In [ ]:
!python src/03_train_val_split.py --window 20

Loading X.npy from: /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/windows_w20
Total samples: 49981

Split fractions:
train_frac = 0.8
val_frac   = 0.1
test_frac  = 0.1

Train / Val / Test sizes:
39984 4998 4999
Saved index files to: /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/splits/splits_w20


In [ ]:
!python src/04_normalize_windows.py --window 20

Loading windows...
Loading splits...
X: (49981, 20, 828)  y: (49981, 1242)
Train/Val/Test: 39984/4998/4999
Computing train-only normalization stats...
Applying normalization and saving...
DONE.
Saved normalized windows to: /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/windows_w20_norm
Saved stats to: /content/drive/MyDrive/Deep Learning/protein_dynamics_project/processed/windows_w20_norm/norm_stats.npz


In [ ]:
!python src/05_train_rnn.py --window 20 --batch 64 --train_fraction 1.0

2026-01-17 09:13:35.439191: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768641215.482817   16079 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768641215.494996   16079 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768641215.528276   16079 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768641215.528311   16079 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768641215.528316   16079 computation_placer.cc:177] computation placer alr

In [ ]:
!python src/05_train_lstm.py --window 10

2026-01-16 12:48:51.631350: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-16 12:48:51.641471: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-16 12:48:51.676934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768567731.733360    3774 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768567731.750723    3774 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768567731.795060    3774 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
!python src/06_train_transformer.py --window 10

2026-01-17 08:39:09.459466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768639149.494032    6743 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768639149.505007    6743 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768639149.529528    6743 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768639149.529577    6743 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768639149.529585    6743 computation_placer.cc:177] computation placer alr

In [ ]:
!python src/07_evaluate_models.py --window 20 --model_path processed/models/rnn_C_w20_best.keras --out_name rnn_C

2026-01-16 18:49:12.363794: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-16 18:49:12.371027: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-16 18:49:12.393600: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768589352.432386   93260 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768589352.443835   93260 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768589352.491112   93260 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
!python - << 'PY'
import numpy as np
from pathlib import Path

PROCESSED = Path("processed")
W = 10  # change to 20 later if needed

X = np.load(PROCESSED / f"windows_w{W}_norm/X.npy", mmap_mode="r")
y = np.load(PROCESSED / f"windows_w{W}_norm/y.npy", mmap_mode="r")
train_idx = np.load(PROCESSED / "splits" / f"splits_w{W}" / "train_idx.npy")

Xtr = np.asarray(X[train_idx], dtype=np.float32).reshape(-1, X.shape[-1])
ytr = np.asarray(y[train_idx], dtype=np.float32)

print("X train mean (avg |mean|):", float(np.mean(np.abs(Xtr.mean(axis=0)))))
print("X train std  (avg |std-1|):", float(np.mean(np.abs(Xtr.std(axis=0) - 1))))
print("y train mean:", float(np.mean(ytr.mean(axis=0))))
print("y train std :", float(np.mean(ytr.std(axis=0))))


/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
X train mean (avg |mean|): 0.013079783879220486
X train std  (avg |std-1|): 0.000421893346356228
y train mean: 6.6144771153631154e-06
y train std : 0.9999998211860657


In [14]:
!python processed/plot_learning_curve.py --history processed/results_step08/tr_C_w10_history.json

Saved learning curve to: processed/results_step08/tr_C_w10_history_learning_curve.jpg
